<a href="https://colab.research.google.com/github/Oktora15/sentimen-analysis-shopee/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from google.colab import files
files.upload()

Saving dataset.csv to dataset (2).csv


{'dataset (2).csv': b'content,score\nbagus apk nya,5\naplikasi inj memiliki program yang nyaman digunakan,5\npengiriman paket nya makin kesini makan lamaaaa,1\n"woii iklan lu di mana"" , buat gaduh aja woii",1\nmantap..klo ada komplen langsung ditangani,5\n"benci bgt sama shopee, setiap ad iklan shopee kita ga mencet apa apa, tbtb kebuka sndiri ke shopee, jd ngelag, akumah gapapa klo sekedar iklan, tp kok tbtb masuk ke dalam shopee nya, sumpah risih banget, aku ud keluar masuk lagi masuk lagi, kesel banget aku tuh...",4\nok,5\nmudah dan asik.,5\naplikasi nya bagus tapi kenapa pas aku mau bayar Qris sulit,5\nsangat membantu,5\nsangat membantu sekali,5\nbagus....kereeen,5\nbagus,5\naplikasi shopi ini sangat sangat bagus dan membantu sekali pokonya mantap de\xe2\x98\xba\xef\xb8\x8f\xf0\x9f\x91\x8d\xf0\x9f\x91\x8d,5\nbgus,4\nbagus.....banyak diskonnya.....,5\n"pengiriman lama, Jabodetabek bisa 2 hari.",1\n"selama ini saya menggunakan aplikasi ini lancar, sempat ada yang mau bobol spy later

In [27]:
import pandas as pd

data = pd.read_csv('dataset.csv')
data.head()

,content,score
0,bagus apk nya,5
1,aplikasi inj memiliki program yang nyaman digu...,5
2,pengiriman paket nya makin kesini makan lamaaaa,1
3,"woii iklan lu di mana"" , buat gaduh aja woii",1
4,mantap..klo ada komplen langsung ditangani,5


In [28]:
def label_sentimen(score):
    if score >= 4:
        return 'positif'
    elif score == 3:
        return 'netral'
    else:
        return 'negatif'

data['label'] = data['score'].apply(label_sentimen)

data[['content', 'score', 'label']].head()

,content,score,label
0,bagus apk nya,5,positif
1,aplikasi inj memiliki program yang nyaman digu...,5,positif
2,pengiriman paket nya makin kesini makan lamaaaa,1,negatif
3,"woii iklan lu di mana"" , buat gaduh aja woii",1,negatif
4,mantap..klo ada komplen langsung ditangani,5,positif


In [29]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['clean_text'] = data['content'].apply(clean_text)

data[['content', 'clean_text']].head()

,content,clean_text
0,bagus apk nya,bagus apk nya
1,aplikasi inj memiliki program yang nyaman digu...,aplikasi inj memiliki program yang nyaman digu...
2,pengiriman paket nya makin kesini makan lamaaaa,pengiriman paket nya makin kesini makan lamaaaa
3,"woii iklan lu di mana"" , buat gaduh aja woii",woii iklan lu di mana buat gaduh aja woii
4,mantap..klo ada komplen langsung ditangani,mantapklo ada komplen langsung ditangani


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9
)

X = tfidf.fit_transform(data['clean_text'])
y = data['label']

print(X.shape)

(3000, 2731)


In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (2400, 2731)
Test: (600, 2731)


In [32]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# model
model_nb = MultinomialNB()
model_lr = LogisticRegression(max_iter=1000, class_weight='balanced')
model_svm = SVC(kernel='linear', class_weight='balanced')

# training
model_nb.fit(X_train, y_train)
model_lr.fit(X_train, y_train)
model_svm.fit(X_train, y_train)

SVC(class_weight='balanced', kernel='linear')

In [33]:
from sklearn.metrics import accuracy_score

y_pred_nb = model_nb.predict(X_test)
y_pred_lr = model_lr.predict(X_test)
y_pred_svm = model_svm.predict(X_test)

print("Naive Bayes:", accuracy_score(y_test, y_pred_nb))
print("Logistic Regression:", accuracy_score(y_test, y_pred_lr))
print("SVM:", accuracy_score(y_test, y_pred_svm))

Naive Bayes: 0.8516666666666667
Logistic Regression: 0.83
SVM: 0.8483333333333334


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# TF-IDF baru (lebih kompleks)
tfidf2 = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.95
)

X2 = tfidf2.fit_transform(data['clean_text'])
y2 = data['label']

# split
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

# model
model_nb2 = MultinomialNB()
model_nb2.fit(X_train2, y_train2)

# evaluasi
y_pred_nb2 = model_nb2.predict(X_test2)

print("Skema 2 (Naive Bayes):", accuracy_score(y_test2, y_pred_nb2))

Skema 2 (Naive Bayes): 0.835


**Analisis Skema 2**

Pada skema ini digunakan TF-IDF dengan ngram_range (1,3) untuk menangkap konteks kata yang lebih luas. Namun, hasil akurasi yang diperoleh sedikit menurun dibandingkan skema pertama.

Hal ini kemungkinan disebabkan oleh meningkatnya kompleksitas fitur yang tidak semuanya relevan, sehingga model mengalami penurunan performa.

In [35]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# CountVectorizer
cv = CountVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X3 = cv.fit_transform(data['clean_text'])
y3 = data['label']

# split
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3, y3,
    test_size=0.2,
    random_state=42,
    stratify=y3
)

# model
model_lr3 = LogisticRegression(max_iter=1000)
model_lr3.fit(X_train3, y_train3)

# evaluasi
y_pred_lr3 = model_lr3.predict(X_test3)

print("Skema 3 (Logistic Regression):", accuracy_score(y_test3, y_pred_lr3))

Skema 3 (Logistic Regression): 0.8683333333333333


**Analisis Skema 3**

Pada skema ini digunakan CountVectorizer untuk mengubah teks menjadi representasi frekuensi kata. Model yang digunakan adalah Logistic Regression.

Hasil yang diperoleh menunjukkan bahwa performa model cukup baik, meskipun masih bergantung pada representasi fitur yang digunakan. Dibandingkan dengan TF-IDF, metode ini lebih sederhana namun tetap mampu menghasilkan performa yang kompetitif.

In [36]:
contoh = ["aplikasinya sangat bagus dan membantu"]
contoh_vector = cv.transform(contoh)

prediksi = model_lr3.predict(contoh_vector)
print("Hasil prediksi:", prediksi)

Hasil prediksi: ['positif']


**Kesimpulan**

Berdasarkan hasil tiga skema eksperimen yang telah dilakukan, diperoleh bahwa model terbaik adalah Logistic Regression pada Skema 3 dengan akurasi sebesar 0.8683.

Skema ini menggunakan CountVectorizer sebagai metode ekstraksi fitur, yang terbukti lebih efektif dibandingkan TF-IDF pada dataset yang digunakan.

Dengan demikian, model Logistic Regression dipilih sebagai model final untuk melakukan inference pada data sentimen.

In [37]:
!pip freeze > requirements.txt